# មេរៀន ០៨ - គំរូរចនាអ្នកតំណាងច្រើន


## ការតម្លើង


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ហេតុអ្វីបានជា ប្រព័ន្ធភាគីច្រើន?

ការងារពិតប្រាកដដូចជាការធ្វើផែនការធ្វើដំណើររាប់បញ្ចូលជំនាញនានា — ដំណើរការផ្គត់ផ្គង់, ជំនាញដឹកនាំក្នុងតំបន់, ការរៀបចំថវិកា, និងផ្សេងទៀត។ ភាគីតែមួយព្យាយាមទទួលខុសត្រូវទាំងអស់ នឹងក្លាយទៅជារឿងពិបាកក្នុងពេលលឿន។

ប្រព័ន្ធភាគីច្រើនដោះស្រាយនេះតាមរយៈ **ការបែងចែកជំនាញ**៖ ភាគីនីមួយៗផ្តោតលើតំបន់ជំនាញមួយ ដើម្បីផលិតលទ្ធផលមានគុណភាពខ្ពស់ជាងអ្នកទូទៅ។ ពួកគេក៏បង្កើន **ភាពអាចបន្ថែម** — អ្នកអាចបន្ថែមភាគីថ្មី (ឧ. អ្នកជំនាញជើងហោះហើរ, អ្នក​អរសាទរម្ហូប) ដោយមិនចាំបាច់សរសេរកូដម្តងទៀតសម្រាប់ដំណើរការដែលមានរួច។ ភាគីទាំងនេះបង្កប់ជាផ្លូវដំណើរការដោយមានរចនាសម្ព័ន្ធ អោយភាគីមួយផ្ទេរបរិបទទៅឱ្យភាគីបន្ទាប់។


## ការបង្កើតភ្នាក់ងារពិសេស


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ការបង្កើតច្រកចរណ៍រៀងរាល់ជំហាន

`WorkflowBuilder` អនុញ្ញាតឱ្យអ្នកភ្ជាប់ភ្នាក់ងារចូលទៅក្នុងក្រាហ្វដែលមានទិសដៅ។ នៅទីនេះយើងបង្កើតបំពង់ពីរជំហានសាមញ្ញមួយ៖ **TravelPlanner** រៀបចំផែនការធ្វើដំណើរ បន្ទាប់មក **TravelConcierge** ពិនិត្យឡើងវិញ និងបង្កើនគុណភាពវា។


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ការបន្ថែមភ្នាក់ងារបន្ថែមទៅក្នុងសីតិ្តការ

មួយក្នុងចំណោមអត្ថប្រយោជន៍ធំពីរបៀប multi-agent គឺភាពងាយស្រួលក្នុងការពង្រីក។ ខាងក្រោមនេះយើងបានបន្ថែមភ្នាក់ងារ **BudgetReviewer** ដែលពិនិត្យផែនការប្រៀបធៀបជាមួយថវិកានៃអ្នកដំណើរ សម្គាល់ធាតុដែលអាចធ្វើឲ្យខ្ពស់លើកំណត់ថវិកា ហើយផ្ដល់យោបល់ជាការជម្រុញការរក្សា ថវិកា។ សីតិ្តការនេះឥឡូវនេះរត់ភ្នាក់ងារបីនៅក្នុងលំដាប់:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## សេចក្ដីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប៖

1. **បង្កើតភ្នាក់ងារពិសេស** — ម្នាក់ម្នាក់មានតួនាទីផ្តោតជាពិសេស (ផែនការ, ពិគ្រោះយោបល់, ពិនិត្យថវិកា)។
2. **ភ្ជាប់ភ្នាក់ងារជារបៀបសំរាប់នីតិកម្មតាមលំដាប់** ដោយប្រើ `WorkflowBuilder` និង `add_edge`។
3. **ផ្សាយចេញស្រកដោត** ពីបណ្ដាញភ្នាក់ងារច្រើន ដើម្បីតាមដានថាភ្នាក់ងារណាកំពុងនិយាយ។
4. **បន្ថែមជួរសកម្មភាពថ្មី** ដោយបន្ថែមភ្នាក់ងារថ្មីទៅច្រកដោយមិនត្រូវផ្លាស់ប្តូរភ្នាក់ងារចាស់ៗ។

រចនាប័ទ្មភ្នាក់ងារច្រើននេះរក្សាឲ្យភ្នាក់ងារនីមួយៗមានភាពសាមញ្ញ ខណៈដែលបង្កើតលទ្ធផលមានភាពကြម និងបានពិនិត្យយ៉ាងពេញលេញជាងភ្នាក់ងារតែម្នាក់អាចធ្វើបាន។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
